In [0]:
from pyspark.sql import functions as F
from datetime import datetime


In [0]:
schema = "key STRING, offset STRING, partition STRING, timestamp STRING, topic STRING, value BINARY"
schema_value = """
  book_id STRING,
  title STRING,
  author STRING,
  price STRING,
  updated STRING
"""

In [0]:
query_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .schema(schema)
    .load("/Volumes/landing/kafka/data/")
    .filter(F.col("topic") == "books")
    .select(
            F.col('value').cast("string").alias('value'),
            F.struct(
                F.col("partition").cast("string").alias("partition"),
                F.col("offset").cast("string").alias("offset"),
                F.col("topic").cast("string").alias("topic"),
                F.col("timestamp").cast("string").alias("timestamp"),
                ).alias('metadata'),
            F.current_timestamp().alias('auditTime')
        )
    .withColumn('value_flat', F.from_json(F.col('value'), schema_value))   
    .select(
        F.col('value_flat.*'),
        F.col('metadata'),
        F.col('auditTime'),
        F.col('value')
    )
)



In [0]:

streaming_query = (
    query_df
        .writeStream
        .format("delta")
        .option("checkpointLocation", "/Volumes/bronze/checkpoint/books/c4")
        .outputMode("append")
        .option("mergeSchema", True)
        .trigger(availableNow=True)
        .toTable("bronze.bookstore.books_TMP4")
)